In [1]:
import os
import numpy as np
import cv2
from PIL import Image, ImageStat
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score
from sklearn.preprocessing import StandardScaler

def estimate_jpeg_quality(img):
    laplacian = cv2.Laplacian(img, cv2.CV_64F)
    laplacian = cv2.convertScaleAbs(laplacian)
    return np.mean(laplacian)

def extract_features(image_path):
    try:
        img = Image.open(image_path)
        img_cv = cv2.imread(image_path)
        if img_cv is None:
            print(f"Warning: Unable to read {image_path} with OpenCV. Skipping...")
            return None
    except:
        print(f"Error: Unable to open {image_path}. Skipping...")
        return None

    features = []

    # 1. Estimated JPEG Quality
    jpeg_quality = estimate_jpeg_quality(img_cv)
    features.append(jpeg_quality)

    # 2. Image sharpness (using variance of Laplacian)
    sharpness = cv2.Laplacian(img_cv, cv2.CV_64F).var()
    features.append(sharpness)

    # 3. RGB channel statistics
    stat = ImageStat.Stat(img)
    for channel in range(3):
        features.extend([stat.mean[channel], stat.rms[channel], stat.var[channel]])

    # 4. Image entropy (measure of image complexity)
    hist = cv2.calcHist([img_cv], [0, 1, 2], None, [8, 8, 8], [0, 256, 0, 256, 0, 256])
    hist = hist / hist.sum()
    entropy = -np.sum(hist * np.log2(hist + 1e-10))
    features.append(entropy)

    # 5. Edge density (using Sobel operator)
    sobel_x = cv2.Sobel(img_cv, cv2.CV_64F, 1, 0, ksize=3)
    sobel_y = cv2.Sobel(img_cv, cv2.CV_64F, 0, 1, ksize=3)
    edge_density = np.mean(np.sqrt(sobel_x**2 + sobel_y**2))
    features.append(edge_density)

    # 6. Color Range (max - min) for each channel
    for i in range(3):
        channel = img_cv[:,:,i]
        features.append(np.max(channel) - np.min(channel))

    return features

# Paths to your dataset
real_path = "C://Users//Sinchan A//Desktop//Internship//vid//real"
fake_path = "C://Users//Sinchan A//Desktop//Internship//vid//fake"


# Lists to store features and labels
X = []
y = []

# Function to safely add features
def add_features(img_path, label):
    features = extract_features(img_path)
    if features is not None:
        X.append(features)
        y.append(label)

# Extract features
for img_name in os.listdir(real_path):
    img_path = os.path.join(real_path, img_name)
    add_features(img_path, 0)  # 0 for real

for img_name in os.listdir(fake_path):
    img_path = os.path.join(fake_path, img_name)
    add_features(img_path, 1)  # 1 for fake

# Convert to numpy arrays (only if we have data)
if X and y:
    X = np.array(X)
    y = np.array(y)

    # Scale the features
    scaler = StandardScaler()
    X = scaler.fit_transform(X)

    # Split into training and testing sets
    X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

    # Define the parameter grid for grid search
    param_grid = {
        'C': [0.1, 1, 10],
        'penalty': ['l1', 'l2'],
        'solver': ['liblinear', 'saga']
    }

    # Create the Logistic Regression model
    lr = LogisticRegression(random_state=42)

    # Perform grid search
    grid_search = GridSearchCV(estimator=lr, param_grid=param_grid, cv=5, n_jobs=-1)
    grid_search.fit(X_train, y_train)

    # Get the best model and its parameters
    best_lr = grid_search.best_estimator_
    print("Best Parameters: ", grid_search.best_params_)

    # Make predictions on test set using the best model
    y_pred = best_lr.predict(X_test)

    # Evaluate the model
    print("\nTest Set Performance (Logistic Regression with Grid Search):")
    print(f"Accuracy: {accuracy_score(y_test, y_pred):.3f}")
    print(f"Precision (Macro): {precision_score(y_test, y_pred, average='macro'):.3f}")
    print(f"Recall (Macro): {recall_score(y_test, y_pred, average='macro'):.3f}")
    print(f"F1-Score (Macro): {f1_score(y_test, y_pred, average='macro'):.3f}")
    print(f"Precision (Weighted): {precision_score(y_test, y_pred, average='weighted'):.3f}")
    print(f"Recall (Weighted): {recall_score(y_test, y_pred, average='weighted'):.3f}")
    print(f"F1-Score (Weighted): {f1_score(y_test, y_pred, average='weighted'):.3f}")


Best Parameters:  {'C': 10, 'penalty': 'l1', 'solver': 'liblinear'}

Test Set Performance (Logistic Regression with Grid Search):
Accuracy: 0.822
Precision (Macro): 0.823
Recall (Macro): 0.823
F1-Score (Macro): 0.822
Precision (Weighted): 0.823
Recall (Weighted): 0.822
F1-Score (Weighted): 0.822


c:\Users\Sinchan A\AppData\Local\Programs\Python\Python312\Lib\site-packages\sklearn\svm\_base.py:1235: ConvergenceWarning: Liblinear failed to converge, increase the number of iterations.
  warnings.warn(
